# Waiter-week model tuning

Tune **Isolation Forest**, **One-Class SVM**, and **LOF** on waiter–week features.

**Objective:** maximize **recall@100** — among all rows, take the top 100 by anomaly score (higher = more anomalous); **recall@100** is the fraction of known frauds that fall in that top set. Same definition as in `models/fit_and_evaluate.py`.

In [21]:
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

ROOT = Path.cwd().parent if Path.cwd().name == "A_tuning" else Path.cwd()
sys.path.insert(0, str(ROOT))

from config import load_data
from models.scaling import scale_features

# Same lists as models/waiter_week_models.py (avoid importing that module: it pulls in models.models).
WAITER_WEEK_FEATURES = [
    "share_loyal_trn",
    "bonusses_accum",
    "trn_per_person",
    "top1_client_trn",
    "top1_client_share_norm",
    "bonusses_used_norm_l",
    "trn_per_person_diff_prev",
    "bonusses_used",
    "top1_client_trn_diff_next",
    "share_new_clients",
    "share_new_clients_norm_diff_prev",
    "top1_client_share_norm",
    "bonusses_trn",
]
WAITER_WEEK_SKEWED = [
    "bonusses_accum",
    "trn_per_person",
    "top1_client_trn",
    "bonusses_used_norm_l",
    "trn_per_person_diff_prev",
    "bonusses_used",
    "top1_client_trn_diff_next",
    "bonusses_trn",
]

MIN_NUM_OF_TRN = 8
NUM_OF_TRN = 1
PLACE_NUM_OF_WAITERS = 1
EXCLUDE_FRAUD_FROM_TRAINING = True
MAX_OCSVM_TRAIN = 4000
RANDOM_STATE = 42

_, _, waiter_week_data, _ = load_data(
    num_of_trn=NUM_OF_TRN,
    place_num_of_waiters=PLACE_NUM_OF_WAITERS,
)
waiter_week_data = waiter_week_data[waiter_week_data["num_of_trn"] >= MIN_NUM_OF_TRN].copy()
y_fraud = waiter_week_data["is_fraud"].astype(int).values
n_fraud = int(y_fraud.sum())
n_total = len(waiter_week_data)
print(f"Samples: {n_total} | Known fraud waiter-weeks: {n_fraud}")
print(f"Features ({len(WAITER_WEEK_FEATURES)}): {WAITER_WEEK_FEATURES}")

if EXCLUDE_FRAUD_FROM_TRAINING:
    train_mask = y_fraud == 0
    fit_subset = waiter_week_data.loc[train_mask]
    X_fit_df, X_eval_df = scale_features(
        data=waiter_week_data,
        features=WAITER_WEEK_FEATURES,
        skewed=WAITER_WEEK_SKEWED,
        scaler_type="standard",
        fit_data=fit_subset,
    )
    X_fit = X_fit_df.to_numpy(dtype=np.float64, copy=False)
    X_eval = X_eval_df.to_numpy(dtype=np.float64, copy=False)
else:
    X_full = scale_features(
        data=waiter_week_data,
        features=WAITER_WEEK_FEATURES,
        skewed=WAITER_WEEK_SKEWED,
        scaler_type="standard",
    )
    X_fit = X_eval = X_full.to_numpy(dtype=np.float64, copy=False)

n_fit = len(X_fit)


K_OPT = 100  # optimize recall @ this rank cutoff


def recall_at_k(anomaly_scores: np.ndarray, fraud_binary: np.ndarray, k: int = K_OPT) -> float:
    """Higher anomaly_scores = more anomalous (same convention as fit_and_evaluate)."""
    fraud_binary = np.asarray(fraud_binary).astype(int)
    n_fraud = int(fraud_binary.sum())
    if n_fraud == 0:
        return float("nan")
    order = np.argsort(-anomaly_scores)
    fraud_mask = fraud_binary.astype(bool)
    top_k = order[: min(k, len(order))]
    return float(fraud_mask[top_k].sum() / n_fraud)


def lof_n_neighbors(requested: int, n_fit_samples: int) -> int:
    return int(min(max(5, requested), max(1, n_fit_samples - 1)))

Samples: 59428 | Known fraud waiter-weeks: 38
Features (13): ['share_loyal_trn', 'bonusses_accum', 'trn_per_person', 'top1_client_trn', 'top1_client_share_norm', 'bonusses_used_norm_l', 'trn_per_person_diff_prev', 'bonusses_used', 'top1_client_trn_diff_next', 'share_new_clients', 'share_new_clients_norm_diff_prev', 'top1_client_share_norm', 'bonusses_trn']


### Isolation Forest

Grid over `n_estimators`, `contamination`, `max_samples`. Initializes `rows`. Metric: **recall@100** from anomaly scores (`-score_samples`).

In [22]:
rows = []

for n_est, cont, ms in product(
    [100, 200, 500],
    [0.0005],
    [1.0, 0.8],
):
    try:
        iso = IsolationForest(
            n_estimators=n_est,
            contamination=cont,
            max_samples=ms,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        iso.fit(X_fit)
        scores = -iso.score_samples(X_eval)
        rows.append(
            {
                "model": "IsolationForest",
                "recall@100": recall_at_k(scores, y_fraud),
                "n_estimators": n_est,
                "contamination": cont,
                "max_samples": ms,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "IsolationForest",
                "recall@100": np.nan,
                "n_estimators": n_est,
                "contamination": cont,
                "max_samples": ms,
                "error": str(exc),
            }
        )

print(f"Isolation Forest: {sum(1 for r in rows if r['model'] == 'IsolationForest')} fits, rows total = {len(rows)}")

Isolation Forest: 6 fits, rows total = 6


### One-Class SVM

Subsample training to `MAX_OCSVM_TRAIN` when needed (same as `fit_and_evaluate`). Metric: **recall@100** from `-decision_function`.

In [15]:
rng = np.random.default_rng(RANDOM_STATE)
if n_fit > MAX_OCSVM_TRAIN:
    idx_sub = rng.choice(n_fit, size=MAX_OCSVM_TRAIN, replace=False)
    X_ocsvm_fit = X_fit[idx_sub]
else:
    X_ocsvm_fit = X_fit

for nu, gamma in product(
    [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],
    ["scale", "auto"],
):
    try:
        ocsvm = OneClassSVM(kernel="rbf", nu=nu, gamma=gamma)
        ocsvm.fit(X_ocsvm_fit)
        scores = -ocsvm.decision_function(X_eval)
        rows.append(
            {
                "model": "OneClassSVM",
                "recall@100": recall_at_k(scores, y_fraud),
                "nu": nu,
                "gamma": gamma,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "OneClassSVM",
                "recall@100": np.nan,
                "nu": nu,
                "gamma": gamma,
                "error": str(exc),
            }
        )

print(f"One-Class SVM: {sum(1 for r in rows if r['model'] == 'OneClassSVM')} fits, rows total = {len(rows)}")

One-Class SVM: 12 fits, rows total = 40


### LOF (Local Outlier Factor)

`novelty=True`: fit on `X_fit`, score on `X_eval`. Metric: **recall@100** from `-score_samples`. Sweeps many **`n_neighbors`** values (each clamped to `[5, n_fit - 1]`).

In [19]:
# n_neighbors is clamped in lof_n_neighbors (min 5, max n_fit - 1); high requests test "use all training neighbors".
for nn_req, cont in product(
    [5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500, 750, 1000, 2000, 5000, 10000],
    # [0.0005, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1],
    [0.01],
):
    nn = lof_n_neighbors(nn_req, n_fit)
    try:
        lof = LocalOutlierFactor(
            n_neighbors=nn,
            contamination=cont,
            metric="minkowski",
            p=2,
            novelty=True,
        )
        lof.fit(X_fit)
        scores = -lof.score_samples(X_eval)
        rows.append(
            {
                "model": "LOF",
                "recall@100": recall_at_k(scores, y_fraud),
                "n_neighbors_requested": nn_req,
                "n_neighbors": nn,
                "contamination": cont,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "LOF",
                "recall@100": np.nan,
                "n_neighbors_requested": nn_req,
                "n_neighbors": nn,
                "contamination": cont,
                "error": str(exc),
            }
        )

print(f"LOF: {sum(1 for r in rows if r['model'] == 'LOF')} fits, rows total = {len(rows)}")

LOF: 23 fits, rows total = 63


### Summary

Build `tuning_results` from all `rows` and rank by **`recall@100`**. Run the three model code cells in order (Isolation Forest first — it initializes `rows`).

In [23]:
tuning_results = pd.DataFrame(rows)
metric = "recall@100"
print(f"Finished {len(tuning_results)} fits (metric: {metric}).\n")

for model_name in ["IsolationForest", "OneClassSVM", "LOF"]:
    sub = tuning_results[tuning_results["model"] == model_name].sort_values(
        metric, ascending=False, na_position="last"
    )
    print(f"=== Top settings: {model_name} (by {metric}) ===")
    print(sub.head(8).to_string(index=False))
    print()

best = tuning_results.sort_values(metric, ascending=False, na_position="last").iloc[0]
print(f"Best overall by {metric}:")
print(best.to_string())

Finished 6 fits (metric: recall@100).

=== Top settings: IsolationForest (by recall@100) ===
          model  recall@100  n_estimators  contamination  max_samples
IsolationForest    0.236842           100         0.0005          1.0
IsolationForest    0.210526           200         0.0005          1.0
IsolationForest    0.184211           500         0.0005          1.0
IsolationForest    0.184211           500         0.0005          0.8
IsolationForest    0.157895           100         0.0005          0.8
IsolationForest    0.157895           200         0.0005          0.8

=== Top settings: OneClassSVM (by recall@100) ===
Empty DataFrame
Columns: [model, recall@100, n_estimators, contamination, max_samples]
Index: []

=== Top settings: LOF (by recall@100) ===
Empty DataFrame
Columns: [model, recall@100, n_estimators, contamination, max_samples]
Index: []

Best overall by recall@100:
model            IsolationForest
recall@100              0.236842
n_estimators                 100
c